In [1]:
import torch

inputs = torch.tensor(
    [[0.72, 0.45, 0.31], # Dream  (x^1)
     [0.75, 0.20, 0.55], # big    (x^2)
     [0.30, 0.80, 0.40], # and    (x^3)
     [0.85, 0.35, 0.60], # work   (x^4)
     [0.55, 0.15, 0.75], # for    (x^5)
     [0.25, 0.20, 0.85]] # it     (x^6)
)

# correspinding words
words = ['Dream', 'big', 'and', 'work', 'for', 'it']

Class for self attention

In [6]:
import torch.nn as nn

class SelfAttention_V2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        # Compute queries, keys, and values
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        d_k = keys.shape[-1]

        # Compute attention scores
        attn_scores = queries @ keys.T

        # Compute attention weights
        attn_weights = torch.softmax(attn_scores / (d_k ** 0.5), dim=-1)

        context_vector = attn_weights @ values
        return context_vector

In [8]:
d_in = inputs.shape[-1]  # Input dimension (3)
d_out = 2  # Output dimension (2)

torch.manual_seed(789)
sa_v2 = SelfAttention_V2(d_in, d_out)

print(sa_v2(inputs))

tensor([[-0.0184,  0.1495],
        [-0.0180,  0.1502],
        [-0.0183,  0.1495],
        [-0.0178,  0.1505],
        [-0.0177,  0.1506],
        [-0.0177,  0.1507]], grad_fn=<MmBackward0>)


Final Attention Weights

In [9]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[0.1582, 0.1729, 0.1456, 0.1732, 0.1771, 0.1730],
        [0.1590, 0.1739, 0.1444, 0.1738, 0.1771, 0.1718],
        [0.1568, 0.1734, 0.1433, 0.1739, 0.1785, 0.1741],
        [0.1570, 0.1750, 0.1403, 0.1751, 0.1793, 0.1732],
        [0.1594, 0.1745, 0.1435, 0.1742, 0.1772, 0.1711],
        [0.1600, 0.1744, 0.1441, 0.1741, 0.1767, 0.1707]],
       grad_fn=<SoftmaxBackward0>)


Lower Triangular matrix (Mask)

In [12]:
context_length = attn_weights.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple) 

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


Applying Mask to attention weights

In [13]:
masked_simple = attn_weights * mask_simple # Element-wise multiplication to apply the mask
print(masked_simple)

tensor([[0.1582, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1590, 0.1739, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1568, 0.1734, 0.1433, 0.0000, 0.0000, 0.0000],
        [0.1570, 0.1750, 0.1403, 0.1751, 0.0000, 0.0000],
        [0.1594, 0.1745, 0.1435, 0.1742, 0.1772, 0.0000],
        [0.1600, 0.1744, 0.1441, 0.1741, 0.1767, 0.1707]],
       grad_fn=<MulBackward0>)


Normalizing Attention weights

In [15]:
row_sum = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sum
print(row_sum)
print(masked_simple_norm)

tensor([[0.1582],
        [0.3329],
        [0.4736],
        [0.6475],
        [0.8289],
        [1.0000]], grad_fn=<SumBackward1>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4777, 0.5223, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3312, 0.3662, 0.3026, 0.0000, 0.0000, 0.0000],
        [0.2425, 0.2703, 0.2167, 0.2704, 0.0000, 0.0000],
        [0.1923, 0.2105, 0.1732, 0.2102, 0.2138, 0.0000],
        [0.1600, 0.1744, 0.1441, 0.1741, 0.1767, 0.1707]],
       grad_fn=<DivBackward0>)


Attention Scores

In [16]:
print(attn_scores)

tensor([[ 0.1278,  0.2536,  0.0104,  0.2559,  0.2873,  0.2540],
        [ 0.1236,  0.2499, -0.0124,  0.2494,  0.2758,  0.2329],
        [ 0.1458,  0.2882,  0.0182,  0.2916,  0.3285,  0.2934],
        [ 0.1517,  0.3052, -0.0074,  0.3055,  0.3393,  0.2904],
        [ 0.1222,  0.2499, -0.0263,  0.2477,  0.2714,  0.2224],
        [ 0.1160,  0.2386, -0.0314,  0.2357,  0.2571,  0.2075]],
       grad_fn=<MmBackward0>)


Upper Triangular Matrix

In [19]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
print(mask)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


Converting 1s to -ve INF

In [24]:
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)  # Fill upper triangular part with -inf mask.bool() converts the mask to a boolean tensor, where True indicates the positions to be masked. The masked_fill function replaces those positions in attn_scores with -inf, effectively masking them out during the softmax operation that follows.
print(masked)

tensor([[ 0.0904,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.0874,  0.1767,    -inf,    -inf,    -inf,    -inf],
        [ 0.1031,  0.2038,  0.0129,    -inf,    -inf,    -inf],
        [ 0.1073,  0.2158, -0.0053,  0.2160,    -inf,    -inf],
        [ 0.0864,  0.1767, -0.0186,  0.1751,  0.1919,    -inf],
        [ 0.0820,  0.1687, -0.0222,  0.1667,  0.1818,  0.1467]],
       grad_fn=<MaskedFillBackward0>)


Applying softmax

In [26]:
attn_weights = torch.softmax(masked/keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4842, 0.5158, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3320, 0.3565, 0.3115, 0.0000, 0.0000, 0.0000],
        [0.2449, 0.2644, 0.2262, 0.2645, 0.0000, 0.0000],
        [0.1947, 0.2075, 0.1808, 0.2073, 0.2098, 0.0000],
        [0.1620, 0.1722, 0.1505, 0.1720, 0.1738, 0.1696]],
       grad_fn=<SoftmaxBackward0>)


Dropout

In [27]:
# Ones Matrix
example = torch.ones(context_length, context_length)
print(example)

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])


Random Dropout with 50% probability

In [28]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example_dropout = dropout(example)
print(example_dropout)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


Attention Weights after Dropout Mask

In [29]:
attn_weights_dropout = dropout(attn_weights)
print(attn_weights_dropout)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7130, 0.6230, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.4524, 0.5290, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.4195, 0.0000],
        [0.3240, 0.3444, 0.0000, 0.3439, 0.3476, 0.3391]],
       grad_fn=<MulBackward0>)
